# STEP 2 — Data Cleaning & Feature Engineering

Pada tahap ini dilakukan proses pembersihan data dan pembuatan fitur tambahan untuk mendukung analisis efektivitas program pemerintah.

Tujuan utama tahap ini adalah:

1. Memuat dataset sintetis yang telah dibuat pada STEP 1.
2. Memeriksa struktur data dan kualitas data.
3. Menghapus duplikasi.
4. Membuat fitur bisnis baru seperti:
   - Budget Efficiency Score
   - Beneficiary Scale
   - Satisfaction Category
   - Budget Tier
   - Program Age
   - Program Maturity
5. Menyimpan dataset final yang siap digunakan untuk EDA, Machine Learning, dan Executive Dashboard.

In [1]:
# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# =========================================================
# 2. LOAD SYNTHETIC DATASET
# =========================================================

# Path relatif dari folder notebooks/
dataset_path = Path("../data/synthetic/synthetic_program_impact_dataset.csv")

# Cek apakah file tersedia
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset not found: {dataset_path}\n"
        "Pastikan STEP 1 sudah berhasil dijalankan."
    )

# Load dataset
df = pd.read_csv(
    dataset_path,
    parse_dates=["start_date"]
)

print("Dataset successfully loaded.")
print("Dataset Shape:", df.shape)

df.head()

Dataset successfully loaded.
Dataset Shape: (10000, 20)


,program_id,program_name,department,district,start_date,budget_allocated,beneficiaries,completion_rate,budget_utilization,satisfaction_score,social_engagement,sentiment_score,reach_rate,cost_per_beneficiary,roi_score,effectiveness_score,impact_category,implementation_status,risk_level,strategic_recommendation
0,PRG-00001,Digital UMKM Empowerment,Dinas Koperasi,Batu,2024-10-08,1551802512,16295,68.66,91.19,4.25,17023,0.703,70.76,95231.82,78.21,78.21,Moderate Impact,In Progress,Medium,Optimize delivery and improve stakeholder enga...
1,PRG-00002,Public WiFi Expansion,DISKOMINFO,Bumiaji,2024-04-14,9463334018,2933,83.97,83.37,3.84,185779,0.456,71.58,3226503.25,78.51,78.51,Moderate Impact,Completed,Medium,Optimize delivery and improve stakeholder enga...
2,PRG-00003,Education Assistance,Dinas Pendidikan,Batu,2024-01-31,902418010,19442,76.86,80.58,3.71,85654,-0.085,55.86,46415.90,69.46,69.46,Low Impact,Completed,High,Conduct a strategic review and redesign the pr...
3,PRG-00004,Tourism Promotion Campaign,Dinas Pariwisata,Bumiaji,2025-09-08,9203905715,1767,99.84,82.56,4.16,4890,0.319,100.00,5208775.16,89.74,89.74,High Impact,Expanded,Low,Scale up program implementation.
4,PRG-00005,Tourism Promotion Campaign,Dinas Pariwisata,Bumiaji,2025-10-27,2301823908,22277,74.80,97.47,4.26,121174,0.291,92.23,103327.37,83.26,83.26,Moderate Impact,In Progress,Medium,Optimize delivery and improve stakeholder enga...


In [3]:
# =========================================================
# 3. DATA OVERVIEW
# =========================================================

# Informasi struktur dataset
print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print(df.info())

# Statistik deskriptif
print("\n" + "=" * 60)
print("DESCRIPTIVE STATISTICS")
print("=" * 60)

df.describe(include="all").T

DATASET INFORMATION
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   program_id                10000 non-null  object        
 1   program_name              10000 non-null  object        
 2   department                10000 non-null  object        
 3   district                  10000 non-null  object        
 4   start_date                10000 non-null  datetime64[ns]
 5   budget_allocated          10000 non-null  int64         
 6   beneficiaries             10000 non-null  int64         
 7   completion_rate           10000 non-null  float64       
 8   budget_utilization        10000 non-null  float64       
 9   satisfaction_score        10000 non-null  float64       
 10  social_engagement         10000 non-null  int64         
 11  sentiment_score           10000 non-null  float64       
 12 

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
program_id,10000,10000,PRG-00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
program_name,10000,10,Digital Literacy Program,1037,NaN,NaN,NaN,NaN,NaN,NaN,NaN
department,10000,7,DISKOMINFO,3059,NaN,NaN,NaN,NaN,NaN,NaN,NaN
district,10000,3,Junrejo,3371,NaN,NaN,NaN,NaN,NaN,NaN,NaN
start_date,10000,NaN,NaN,NaN,2025-01-02 21:19:09.120000,2024-01-01 00:00:00,2024-07-01 00:00:00,2025-01-06 00:00:00,2025-07-07 00:00:00,2025-12-31 00:00:00,NaN
budget_allocated,10000.0,NaN,NaN,NaN,5178826157.4851,500571208.0,2790956816.75,5173063904.0,7526619546.5,9999260135.0,2736200509.877151
beneficiaries,10000.0,NaN,NaN,NaN,25360.7972,501.0,12842.0,25308.5,37919.0,49986.0,14377.382815
completion_rate,10000.0,NaN,NaN,NaN,81.800216,39.4,74.11,82.02,90.2125,100.0,11.243617
budget_utilization,10000.0,NaN,NaN,NaN,87.333954,51.63,81.13,87.86,94.62,100.0,9.048871
satisfaction_score,10000.0,NaN,NaN,NaN,4.092763,1.92,3.73,4.1,4.47,5.0,0.522988


In [4]:
# =========================================================
# 4. MISSING VALUE CHECK
# =========================================================

missing_values = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

print("=" * 60)
print("MISSING VALUE SUMMARY")
print("=" * 60)

if missing_values.sum() == 0:
    print("No missing values detected. ✅")
else:
    print(missing_values[missing_values > 0])

MISSING VALUE SUMMARY
No missing values detected. ✅


In [5]:
# =========================================================
# 5. REMOVE DUPLICATES
# =========================================================

before_rows = len(df)

df = df.drop_duplicates()

after_rows = len(df)
removed_rows = before_rows - after_rows

print("=" * 60)
print("DUPLICATE REMOVAL")
print("=" * 60)
print(f"Rows before cleaning : {before_rows:,}")
print(f"Rows after cleaning  : {after_rows:,}")
print(f"Duplicates removed   : {removed_rows:,}")

DUPLICATE REMOVAL
Rows before cleaning : 10,000
Rows after cleaning  : 10,000
Duplicates removed   : 0


In [7]:
# =========================================================
# 6. FEATURE ENGINEERING
# =========================================================

# ---------------------------------------------------------
# Budget Efficiency Score
# Mengukur efektivitas relatif terhadap tingkat utilisasi anggaran
# ---------------------------------------------------------
df["budget_efficiency_score"] = (
    df["effectiveness_score"] /
    (df["budget_utilization"] / 100)
).clip(0, 150)

# ---------------------------------------------------------
# Beneficiary Scale
# Mengelompokkan program berdasarkan jumlah penerima manfaat
# ---------------------------------------------------------
df["beneficiary_scale"] = pd.cut(
    df["beneficiaries"],
    bins=[0, 5000, 20000, 50000],
    labels=[
        "Small Scale",
        "Medium Scale",
        "Large Scale"
    ]
)

# ---------------------------------------------------------
# Satisfaction Category
# Kategorisasi tingkat kepuasan masyarakat
# ---------------------------------------------------------
df["satisfaction_category"] = pd.cut(
    df["satisfaction_score"],
    bins=[0, 3.0, 4.0, 5.0],
    labels=[
        "Low",
        "Moderate",
        "High"
    ],
    include_lowest=True
)

# ---------------------------------------------------------
# Budget Tier
# Mengelompokkan program ke dalam kuartil anggaran
# ---------------------------------------------------------
df["budget_tier"] = pd.qcut(
    df["budget_allocated"],
    q=4,
    labels=[
        "Tier 1",
        "Tier 2",
        "Tier 3",
        "Tier 4"
    ]
)

# ---------------------------------------------------------
# Program Age (Days)
# Lama program sejak tanggal mulai
# ---------------------------------------------------------
df["program_age_days"] = (
    pd.Timestamp.today().normalize() -
    df["start_date"]
).dt.days

# ---------------------------------------------------------
# Program Maturity
# Tahap kematangan program
# ---------------------------------------------------------
df["program_maturity"] = pd.cut(
    df["program_age_days"],
    bins=[0, 180, 365, 730, 5000],
    labels=[
        "Early Stage",
        "Growing",
        "Established",
        "Mature"
    ],
    include_lowest=True
)

print("Feature engineering completed successfully. ✅")
print("New Dataset Shape:", df.shape)

df.head()

Feature engineering completed successfully. ✅
New Dataset Shape: (10000, 26)


,program_id,program_name,department,district,start_date,budget_allocated,beneficiaries,completion_rate,budget_utilization,satisfaction_score,...,impact_category,implementation_status,risk_level,strategic_recommendation,budget_efficiency_score,beneficiary_scale,satisfaction_category,budget_tier,program_age_days,program_maturity
0,PRG-00001,Digital UMKM Empowerment,Dinas Koperasi,Batu,2024-10-08,1551802512,16295,68.66,91.19,4.25,...,Moderate Impact,In Progress,Medium,Optimize delivery and improve stakeholder enga...,85.765983,Medium Scale,High,Tier 1,581,Established
1,PRG-00002,Public WiFi Expansion,DISKOMINFO,Bumiaji,2024-04-14,9463334018,2933,83.97,83.37,3.84,...,Moderate Impact,Completed,Medium,Optimize delivery and improve stakeholder enga...,94.170565,Small Scale,Moderate,Tier 4,758,Mature
2,PRG-00003,Education Assistance,Dinas Pendidikan,Batu,2024-01-31,902418010,19442,76.86,80.58,3.71,...,Low Impact,Completed,High,Conduct a strategic review and redesign the pr...,86.200050,Medium Scale,Moderate,Tier 1,832,Mature
3,PRG-00004,Tourism Promotion Campaign,Dinas Pariwisata,Bumiaji,2025-09-08,9203905715,1767,99.84,82.56,4.16,...,High Impact,Expanded,Low,Scale up program implementation.,108.696705,Small Scale,High,Tier 4,246,Growing
4,PRG-00005,Tourism Promotion Campaign,Dinas Pariwisata,Bumiaji,2025-10-27,2301823908,22277,74.80,97.47,4.26,...,Moderate Impact,In Progress,Medium,Optimize delivery and improve stakeholder enga...,85.421155,Large Scale,High,Tier 1,197,Growing


In [8]:
# =========================================================
# 7. PREVIEW ENGINEERED FEATURES
# =========================================================

new_features = [
    "budget_efficiency_score",
    "beneficiary_scale",
    "satisfaction_category",
    "budget_tier",
    "program_age_days",
    "program_maturity"
]

df[new_features].head(10)

,budget_efficiency_score,beneficiary_scale,satisfaction_category,budget_tier,program_age_days,program_maturity
0,85.765983,Medium Scale,High,Tier 1,581,Established
1,94.170565,Small Scale,Moderate,Tier 4,758,Mature
2,86.200050,Medium Scale,Moderate,Tier 1,832,Mature
3,108.696705,Small Scale,High,Tier 4,246,Growing
4,85.421155,Large Scale,High,Tier 1,197,Growing
5,84.838408,Medium Scale,Moderate,Tier 2,578,Established
6,93.192367,Large Scale,High,Tier 1,578,Established
7,88.977160,Large Scale,High,Tier 1,768,Mature
8,93.356553,Medium Scale,Moderate,Tier 4,244,Growing
9,94.286047,Medium Scale,Moderate,Tier 1,735,Mature


In [9]:
# =========================================================
# 8. VALIDATE ENGINEERED FEATURES
# =========================================================

print("=" * 60)
print("ENGINEERED FEATURE SUMMARY")
print("=" * 60)

# Numerical summary
print("\nNumerical Features:")
print(
    df[
        [
            "budget_efficiency_score",
            "program_age_days"
        ]
    ].describe().round(2)
)

# Categorical distributions
categorical_features = [
    "beneficiary_scale",
    "satisfaction_category",
    "budget_tier",
    "program_maturity"
]

for feature in categorical_features:
    print("\n" + "-" * 60)
    print(feature.upper())
    print("-" * 60)
    print(
        df[feature]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .astype(str) + "%"
    )

ENGINEERED FEATURE SUMMARY

Numerical Features:
       budget_efficiency_score  program_age_days
count                 10000.00          10000.00
mean                     93.55            494.11
std                      10.60            212.17
min                      64.24            132.00
25%                      85.97            309.00
50%                      92.29            491.00
75%                      99.74            680.00
max                     150.00            862.00

------------------------------------------------------------
BENEFICIARY_SCALE
------------------------------------------------------------
beneficiary_scale
Large Scale     60.94%
Medium Scale    30.01%
Small Scale      9.05%
Name: proportion, dtype: object

------------------------------------------------------------
SATISFACTION_CATEGORY
------------------------------------------------------------
satisfaction_category
High        57.19%
Moderate    40.68%
Low          2.13%
Name: proportion, dtype: ob

In [10]:
# =========================================================
# 9. SAVE CLEANED DATASET
# =========================================================

# Create output directory
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Output file path
output_path = output_dir / "program_impact_cleaned.csv"

# Save dataset
df.to_csv(output_path, index=False)

print("=" * 60)
print("CLEANED DATASET SAVED SUCCESSFULLY")
print("=" * 60)
print(f"Output Path : {output_path}")
print(f"Dataset Shape: {df.shape}")

CLEANED DATASET SAVED SUCCESSFULLY
Output Path : ..\data\processed\program_impact_cleaned.csv
Dataset Shape: (10000, 26)


In [11]:
# =========================================================
# 10. FINAL PREVIEW
# =========================================================

df.head()

,program_id,program_name,department,district,start_date,budget_allocated,beneficiaries,completion_rate,budget_utilization,satisfaction_score,...,impact_category,implementation_status,risk_level,strategic_recommendation,budget_efficiency_score,beneficiary_scale,satisfaction_category,budget_tier,program_age_days,program_maturity
0,PRG-00001,Digital UMKM Empowerment,Dinas Koperasi,Batu,2024-10-08,1551802512,16295,68.66,91.19,4.25,...,Moderate Impact,In Progress,Medium,Optimize delivery and improve stakeholder enga...,85.765983,Medium Scale,High,Tier 1,581,Established
1,PRG-00002,Public WiFi Expansion,DISKOMINFO,Bumiaji,2024-04-14,9463334018,2933,83.97,83.37,3.84,...,Moderate Impact,Completed,Medium,Optimize delivery and improve stakeholder enga...,94.170565,Small Scale,Moderate,Tier 4,758,Mature
2,PRG-00003,Education Assistance,Dinas Pendidikan,Batu,2024-01-31,902418010,19442,76.86,80.58,3.71,...,Low Impact,Completed,High,Conduct a strategic review and redesign the pr...,86.200050,Medium Scale,Moderate,Tier 1,832,Mature
3,PRG-00004,Tourism Promotion Campaign,Dinas Pariwisata,Bumiaji,2025-09-08,9203905715,1767,99.84,82.56,4.16,...,High Impact,Expanded,Low,Scale up program implementation.,108.696705,Small Scale,High,Tier 4,246,Growing
4,PRG-00005,Tourism Promotion Campaign,Dinas Pariwisata,Bumiaji,2025-10-27,2301823908,22277,74.80,97.47,4.26,...,Moderate Impact,In Progress,Medium,Optimize delivery and improve stakeholder enga...,85.421155,Large Scale,High,Tier 1,197,Growing
